Cell 1: Code (Install Dependencies)

In [ ]:
!pip install timm pytorch-lightning wandb opencv-python scikit-learn seaborn
import torch
torch.cuda.is_available()  # Should return True on A100

Cell 2: Mount your dataset directory and update drive_path in the next cell.

Cell 3: Code (Imports)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset, random_split
import torchvision.transforms as transforms
from timm import create_model
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.loggers import WandbLogger
import wandb
import cv2
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight
from torch.cuda.amp import autocast, GradScaler  # For AMP
import os
from glob import glob
from PIL import Image

Cell 4: Code (Dataset Class and Loading)

In [ ]:
import os, cv2, torch, numpy as np
from glob import glob
from PIL import Image
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset, random_split
import torchvision.transforms as transforms
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)


def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return torch.tensor([]), torch.tensor([])
    return torch.utils.data.dataloader.default_collate(batch)


class VideoDataset(Dataset):
    """
    Reads ONE frame from each clip at a configurable position.
    sample='middle' picks the middle frame (default; the discriminating
    moment for a stroke). sample='first' reproduces the old behaviour.
    """
    def __init__(self, video_paths, labels, transform=None, sample='middle'):
        self.video_paths = video_paths
        self.labels = labels
        self.transform = transform
        self.sample = sample

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        path, label = self.video_paths[idx], self.labels[idx]
        try:
            cap = cv2.VideoCapture(path)
            n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if n_frames <= 0:
                cap.release()
                return None

            if self.sample == 'middle':
                target = n_frames // 2
            elif self.sample == 'first':
                target = 0
            else:
                target = np.random.randint(max(1, n_frames))

            cap.set(cv2.CAP_PROP_POS_FRAMES, target)
            ret, frame = cap.read()
            cap.release()
            if not ret:
                # fall back to first frame if seek failed
                cap = cv2.VideoCapture(path)
                ret, frame = cap.read()
                cap.release()
                if not ret:
                    return None

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(frame)
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error on {path}: {e}")
            return None


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.33)),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# --- discover classes & paths
drive_path = '/path/to/your/dataset/'
class_names = sorted([d for d in os.listdir(drive_path)
                      if os.path.isdir(os.path.join(drive_path, d))])
class_to_idx = {cls: i for i, cls in enumerate(class_names)}

video_paths, labels = [], []
for cls in class_names:
    cls_videos = glob(os.path.join(drive_path, cls, '*.mp4'))
    video_paths.extend(cls_videos)
    labels.extend([class_to_idx[cls]] * len(cls_videos))

# remove Test_Dataset folder if present
if 'Test_Dataset' in class_names:
    bad = class_names.index('Test_Dataset')
    keep = [i for i, l in enumerate(labels) if l != bad]
    video_paths = [video_paths[i] for i in keep]
    old_labels = [labels[i] for i in keep]
    idx_to_class = {i: c for c, i in class_to_idx.items()}
    class_names.remove('Test_Dataset')
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    labels = [class_to_idx[idx_to_class[l]] for l in old_labels]

num_classes = len(class_names)
print(f"Num classes: {num_classes}")
print(f"Total clips: {len(video_paths)}")

# --- reproducible split
dataset = list(zip(video_paths, labels))
gen = torch.Generator().manual_seed(SEED)
n = len(dataset)
n_train = int(0.8 * n); n_val = int(0.1 * n); n_test = n - n_train - n_val
train_data, val_data, test_data = random_split(dataset, [n_train, n_val, n_test], generator=gen)

train_dataset = VideoDataset([p for p, l in train_data], [l for p, l in train_data],
                             transform=train_transform, sample='middle')
val_dataset   = VideoDataset([p for p, l in val_data],   [l for p, l in val_data],
                             transform=val_transform,   sample='middle')
test_dataset  = VideoDataset([p for p, l in test_data],  [l for p, l in test_data],
                             transform=val_transform,   sample='middle')

train_labels = [l for p, l in train_data]
class_weights = compute_class_weight('balanced',
                                     classes=np.unique(train_labels),
                                     y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).cuda()

sampler_weights = [class_weights[label].item() for label in train_labels]
sampler = WeightedRandomSampler(weights=sampler_weights,
                                num_samples=len(train_labels),
                                replacement=True)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=8, pin_memory=True, persistent_workers=True,
                          collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=8, pin_memory=True, persistent_workers=True,
                          collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=8, pin_memory=True, persistent_workers=True,
                          collate_fn=collate_fn)

Cell 5: Code (Model)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pytorch_lightning as pl
from timm import create_model


class FocalLoss(nn.Module):
    """Optional focal loss with optional class weights and label smoothing."""
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight,
                             label_smoothing=self.label_smoothing,
                             reduction='none')
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


class GenericClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.3):
        super().__init__()
        self.model = create_model(model_name, pretrained=True, num_classes=num_classes)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.model(x))


class LitClassifier(pl.LightningModule):
    """
    Works with any timm backbone. Correctly accumulates per-epoch
    metrics in Lightning 2.x using explicit list attributes (the bug
    your original LitSwin had).
    """
    def __init__(self, model_name, num_classes, class_weights,
                 lr=1e-4, max_epochs=30, steps_per_epoch=None,
                 use_focal=False, gamma=2.0, label_smoothing=0.1,
                 weight_decay=1e-2):
        super().__init__()
        self.save_hyperparameters(ignore=['class_weights'])
        self.model = GenericClassifier(model_name, num_classes)
        if use_focal:
            self.criterion = FocalLoss(weight=class_weights, gamma=gamma,
                                       label_smoothing=label_smoothing)
        else:
            self.criterion = nn.CrossEntropyLoss(weight=class_weights,
                                                 label_smoothing=label_smoothing)
        self.lr = lr
        self.max_epochs = max_epochs
        self.steps_per_epoch = steps_per_epoch
        self.weight_decay = weight_decay
        self._val, self._test = [], []qq3

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.criterion(self(x), y)
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        self.log('val_loss', loss, on_epoch=True, prog_bar=True)
        self._val.append({'logits': logits.detach().cpu(), 'targets': y.detach().cpu()})

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        self.log('test_loss', loss)
        self._test.append({'logits': logits.detach().cpu(), 'targets': y.detach().cpu()})

    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        x, y = batch
        return self(x), y

    def _topk_metrics(self, outputs, prefix):
        logits = torch.cat([o['logits'] for o in outputs])
        targets = torch.cat([o['targets'] for o in outputs])
        preds = logits.argmax(dim=1)
        acc1 = (preds == targets).float().mean()
        _, top3 = torch.topk(logits, 3, dim=1)
        _, top5 = torch.topk(logits, 5, dim=1)
        acc3 = (top3 == targets.unsqueeze(1)).any(dim=1).float().mean()
        acc5 = (top5 == targets.unsqueeze(1)).any(dim=1).float().mean()
        self.log_dict({f'{prefix}_acc1': acc1,
                       f'{prefix}_acc3': acc3,
                       f'{prefix}_acc5': acc5}, prog_bar=True)

    def on_validation_epoch_end(self):
        if self._val:
            self._topk_metrics(self._val, 'val')
            self._val.clear()

    def on_test_epoch_end(self):
        if self._test:
            self._topk_metrics(self._test, 'test')
            self._test.clear()

    def configure_optimizers(self):
        opt = optim.AdamW(self.parameters(), lr=self.lr,
                          weight_decay=self.weight_decay)
        steps = self.steps_per_epoch or len(train_loader)
        sched = optim.lr_scheduler.OneCycleLR(
            opt, max_lr=self.lr,
            epochs=self.max_epochs,
            steps_per_epoch=steps)
        return {'optimizer': opt,
                'lr_scheduler': {'scheduler': sched, 'interval': 'step'}}

Cell 6: Code (Trainer)

In [ ]:
import os, json
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import (ModelCheckpoint,
                                         LearningRateMonitor,
                                         TQDMProgressBar)

BACKBONES = [
    ('resnet50',                     'ResNet-50'),
    ('efficientnet_b3',              'EfficientNet-B3'),
    ('convnext_tiny',                'ConvNeXt-Tiny'),
    ('swin_tiny_patch4_window7_224', 'Swin-Tiny (ours)'),
]
EPOCHS = 30
USE_FOCAL = False          # set True for the focal-loss recipe variant
LR = 1e-4

os.makedirs('runs', exist_ok=True)
results = {}

for model_name, display in BACKBONES:
    print(f"\n{'='*72}\n>>> Training {display} ({model_name})\n{'='*72}")
    pl.seed_everything(SEED, workers=True)

    model = LitClassifier(
        model_name=model_name,
        num_classes=num_classes,
        class_weights=class_weights,
        lr=LR,
        max_epochs=EPOCHS,
        steps_per_epoch=len(train_loader),
        use_focal=USE_FOCAL,
        label_smoothing=0.1,
        weight_decay=1e-2,
    )

    out_dir = f'runs/{model_name}'
    os.makedirs(out_dir, exist_ok=True)

    ckpt_cb = ModelCheckpoint(dirpath=out_dir, filename='best',
                              monitor='val_acc1', mode='max', save_top_k=1)
    trainer = pl.Trainer(
        max_epochs=EPOCHS,
        accelerator='gpu', devices=1,
        precision='16-mixed',
        logger=TensorBoardLogger('tb_logs', name=model_name),
        callbacks=[ckpt_cb, LearningRateMonitor(logging_interval='step'),
                   TQDMProgressBar(refresh_rate=20)],
        benchmark=True,
        deterministic=False,
    )
    trainer.fit(model, train_loader, val_loader)

    # evaluate on test set using best checkpoint
    preds = trainer.predict(model, test_loader, ckpt_path='best')
    logits = torch.cat([p[0] for p in preds]).cpu()
    targets = torch.cat([p[1] for p in preds]).cpu()

    pred_top1 = logits.argmax(dim=1)
    top1 = (pred_top1 == targets).float().mean().item()
    _, top3 = torch.topk(logits, 3, dim=1)
    _, top5 = torch.topk(logits, 5, dim=1)
    top3_acc = (top3 == targets.unsqueeze(1)).any(dim=1).float().mean().item()
    top5_acc = (top5 == targets.unsqueeze(1)).any(dim=1).float().mean().item()
    n_params = sum(p.numel() for p in model.parameters()) / 1e6

    results[display] = {
        'model_name': model_name,
        'params_M': round(n_params, 1),
        'top1': round(top1 * 100, 2),
        'top3': round(top3_acc * 100, 2),
        'top5': round(top5_acc * 100, 2),
    }
    torch.save({'logits': logits, 'targets': targets,
                'class_names': class_names, 'metrics': results[display]},
               f'{out_dir}/test_outputs.pt')

    print(f"\n{display}  Top-1={top1*100:5.2f}%  Top-3={top3_acc*100:5.2f}%  "
          f"Top-5={top5_acc*100:5.2f}%  ({n_params:.1f}M params)")

    del model, trainer
    torch.cuda.empty_cache()

with open('runs/comparison_results.json', 'w') as f:
    json.dump(results, f, indent=2)

# print the comparative table for the dissertation
print("\n\n" + "="*72)
print("COMPARATIVE EVALUATION  (paste this into the dissertation)")
print("="*72)
print(f"{'Model':<22} {'Params(M)':>10} {'Top-1':>9} {'Top-3':>9} {'Top-5':>9}")
print("-"*65)
for name, r in results.items():
    print(f"{name:<22} {r['params_M']:>10.1f} "
          f"{r['top1']:>8.2f}% {r['top3']:>8.2f}% {r['top5']:>8.2f}%")

Cell 7: Code (Evaluation)

In [ ]:
import torch, numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# pick the best model from your comparative table
BEST_MODEL = 'swin_tiny_patch4_window7_224'

data = torch.load(f'runs/{BEST_MODEL}/test_outputs.pt')
logits = data['logits'].numpy()
targets = data['targets'].numpy()
preds = logits.argmax(axis=1)
class_names = data['class_names']

# 1. Raw confusion matrix
cm = confusion_matrix(targets, preds)
plt.figure(figsize=(15, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'Confusion Matrix — {BEST_MODEL}', fontsize=16)
plt.xlabel('Predicted Label'); plt.ylabel('True Label')
plt.tight_layout()
plt.savefig(f'runs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# 2. Row-normalised confusion matrix (this is the one for the dissertation)
row_sums = cm.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
cm_norm = cm / row_sums
plt.figure(figsize=(15, 12))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1,
            xticklabels=class_names, yticklabels=class_names)
plt.title('Row-Normalised Confusion Matrix', fontsize=16)
plt.xlabel('Predicted Label'); plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('runs/confusion_matrix_normalised.png', dpi=150, bbox_inches='tight')
plt.show()

# 3. Classification report (precision/recall/F1 per class)
print("=== CLASSIFICATION REPORT ===")
print(classification_report(targets, preds, target_names=class_names, digits=4))

# 4. Per-class top-k table
print("\n=== PER-CLASS TOP-k ACCURACY ===")
print(f"{'Class':<26} {'N':>5} {'Top-1':>9} {'Top-3':>9} {'Top-5':>9}")
print("-"*65)
for i, cls in enumerate(class_names):
    mask = targets == i
    if mask.sum() == 0:
        continue
    cl = logits[mask]; ct = targets[mask]
    a1 = (cl.argmax(axis=1) == ct).mean() * 100
    t3 = (-cl).argsort(axis=1)[:, :3]
    t5 = (-cl).argsort(axis=1)[:, :5]
    a3 = np.mean([ct[j] in t3[j] for j in range(len(ct))]) * 100
    a5 = np.mean([ct[j] in t5[j] for j in range(len(ct))]) * 100
    print(f"{cls:<26} {mask.sum():>5d} {a1:>8.2f}% {a3:>8.2f}% {a5:>8.2f}%")

# 5. Most-confused pairs (for the qualitative section)
print("\n=== MOST-CONFUSED PAIRS (off-diagonal) ===")
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
flat = [(i, j, cm_off[i, j]) for i in range(len(class_names))
        for j in range(len(class_names)) if cm_off[i, j] > 0]
flat.sort(key=lambda x: -x[2])
for i, j, n in flat[:15]:
    pct = 100 * n / cm[i].sum()
    print(f"  True={class_names[i]:<22} -> Pred={class_names[j]:<22}  "
          f"{n:>4} ({pct:.1f}% of class)")